# CLINICAL TRIALS RISK ANALYSIS
---
## 01 - Data Collection
This notebook performs the first step of data pipeline: **retrieving raw clinical trial records using the modern ClinicalTrials.gov JSON API (API v2)**.

### Objective  
The objective of this step is to programmatically download the **first 5,000 clinical trial study records**, preserve their full hierarchical structure, and save them as **raw JSON files** for reproducibility.  
These unprocessed records will later be flattened, cleaned, and transformed into a tabular, ML-ready dataset in the preprocessing stage.

### Why collect raw JSON?  
ClinicalTrials.gov API v2 provides a modernized schema where each study is represented as a deeply nested JSON object containing:

- **protocolSection** $\rightarrow$ primary study information  
  - **identificationModule** $\rightarrow$ NCT ID, titles, organization  
  - **statusModule** $\rightarrow$ recruitment status, key study dates  
  - **designModule** $\rightarrow$ study design, phase, allocation, model  
  - **conditionsModule** $\rightarrow$ conditions, keywords  
  - **armsInterventionsModule** $\rightarrow$ arms, interventions, drug/device info  
  - **outcomesModule** $\rightarrow$ primary/secondary outcomes  
  - **descriptionModule** $\rightarrow$ brief and detailed summaries  
  - **sponsorCollaboratorsModule** $\rightarrow$ lead sponsor, collaborators  
  - **eligibilityModule** $\rightarrow$ inclusion/exclusion criteria, sex, age  
  - **contactsLocationsModule** $\rightarrow$ study officials, locations  
  - **oversightModule** $\rightarrow$ oversight and data monitoring committee info  
  - **referencesModule** $\rightarrow$ PMIDs and literature references  
- **derivedSection** $\rightarrow$ standardized MeSH terms & curated metadata  
- **hasResults** $\rightarrow$ boolean flag indicating whether the study has posted results  

Working directly with the raw JSON ensures that no structural information is lost before feature extraction.

### Output  
Running this notebook will generate: `data/raw/clinical_trials_5000.json` 
containing approximately **5,000 complete study objects**, which forms the foundation for downstream preprocessing, EDA, and risk-prediction modeling.

### Imports & Config

In [3]:
import sys
import os

# Path to project root (folder that contains "src/")
project_root = os.path.abspath("..")
sys.path.append(project_root)

masked = project_root.replace(os.path.expanduser("~"), "~")
print("Project root added:", masked)

Project root added: ~/Clinical-Trial-Failure-Prediction


In [4]:
import json
import requests

from src.data.data_loader import ClinicalTrialsAPI

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
OUTPUT_PATH = "../data/raw/clinical_trials_5000.json"

### Smoke Test: Fetch 1 Record

In [5]:
# Quick test request to check API availability & structure
response = requests.get(BASE_URL, params={"pageSize": 1})
response.raise_for_status()

sample = response.json()
sample.keys(), len(sample["studies"])

(dict_keys(['studies', 'nextPageToken']), 1)

### Inspect Structure of a Study (Schema Exploration)

In [6]:
sample_study = sample["studies"][0]

protocol = sample_study.get("protocolSection", {})

print("protocolSection keys:")
print(list(protocol.keys()))

ident = protocol.get("identificationModule", {})
status = protocol.get("statusModule", {})
design = protocol.get("designModule", {})

print("\nIdentification Module:")
print(json.dumps(ident, indent=2))

print("\nStatus Module:")
print(json.dumps(status, indent=2))

print("\nDesign Module:")
print(json.dumps(design, indent=2))

protocolSection keys:
['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'oversightModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule']

Identification Module:
{
  "nctId": "NCT05800535",
  "orgStudyIdInfo": {
    "id": "813"
  },
  "secondaryIdInfos": [
    {
      "id": "69HCL23_0177",
      "type": "OTHER",
      "domain": "HCL"
    }
  ],
  "organization": {
    "fullName": "Hospices Civils de Lyon",
    "class": "OTHER"
  },
  "briefTitle": "JUMP Prevalence of Sarcopenia and Chemobrain in Post-cancer Patients",
  "officialTitle": "Prevalence of Sarcopenia and Chemobrain in Post-cancer Patients",
  "acronym": "JUMP"
}

Status Module:
{
  "statusVerifiedDate": "2025-11",
  "overallStatus": "RECRUITING",
  "expandedAccessInfo": {
    "hasExpandedAccess": false
  },
  "startDateStruct": {
    "date": "2025-02-10",
    "type": "ACTUAL"
  },
  "primaryCompletionD

### Initialize API Loader (from data_loader.py)

In [7]:
api = ClinicalTrialsAPI(page_size=100, sleep=0.15, max_retries=3)

api 

### Fetch 5000 Studies

In [8]:
studies = api.fetch_n_studies(n=5000)
len(studies)

[INFO] Fetching up to 5000 studies...
[INFO] Retrieved batch with 100 studies (total: 100)
[INFO] Retrieved batch with 100 studies (total: 200)
[INFO] Retrieved batch with 100 studies (total: 300)
[INFO] Retrieved batch with 100 studies (total: 400)
[INFO] Retrieved batch with 100 studies (total: 500)
[INFO] Retrieved batch with 100 studies (total: 600)
[INFO] Retrieved batch with 100 studies (total: 700)
[INFO] Retrieved batch with 100 studies (total: 800)
[INFO] Retrieved batch with 100 studies (total: 900)
[INFO] Retrieved batch with 100 studies (total: 1000)
[INFO] Retrieved batch with 100 studies (total: 1100)
[INFO] Retrieved batch with 100 studies (total: 1200)
[INFO] Retrieved batch with 100 studies (total: 1300)
[INFO] Retrieved batch with 100 studies (total: 1400)
[INFO] Retrieved batch with 100 studies (total: 1500)
[INFO] Retrieved batch with 100 studies (total: 1600)
[INFO] Retrieved batch with 100 studies (total: 1700)
[INFO] Retrieved batch with 100 studies (total: 1800)

5000

### Quick Sanity Check

In [9]:
# Show first 10 NCT IDs
ids = [
    s.get("protocolSection", {})
     .get("identificationModule", {})
     .get("nctId")
    for s in studies[:10]
]

ids

['NCT05800535',
 'NCT03207035',
 'NCT04109235',
 'NCT06449235',
 'NCT00358735',
 'NCT03905135',
 'NCT07112235',
 'NCT06502535',
 'NCT02813135',
 'NCT01757535']

### Save Raw JSON

In [10]:
api.save_json(studies, OUTPUT_PATH)

[INFO] JSON saved to: ../data/raw/clinical_trials_5000.json


### Validate Output File

In [11]:
size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"Saved file size: {size_mb:.2f} MB")

Saved file size: 130.73 MB
